# Week 2: Machine Learning Model Training

**Team:** Juan and Nathan  
**Goal:** Build models that beat hospital baseline of 83.9%  
**Approach:** Train 3 models, pick the best one

## 1. Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

In [ ]:
# Load the cleaned dataset from Week 1
try:
    # Try multiple file locations
    for filepath in ['clean_heart_failure_data.csv', '../original.csv', '../training_data.csv']:
        try:
            df = pd.read_csv(filepath)
            print(f"Dataset loaded from: {filepath}")
            break
        except FileNotFoundError:
            continue
    
    print(f"Dataset shape: {df.shape}")
    print(f"Features: {list(df.columns)}")
    
except Exception as e:
    print(f"Error loading dataset: {e}")

## 2. Prepare Data for Machine Learning

In [ ]:
# Separate features and target
print("PREPARING DATA FOR MODELING")
print("="*40)

if 'DEATH_EVENT' not in df.columns:
    print("Error: Target variable 'DEATH_EVENT' not found")
else:
    X = df.drop('DEATH_EVENT', axis=1)
    y = df['DEATH_EVENT']
    
    print(f"Features shape: {X.shape}")
    print(f"Target shape: {y.shape}")
    print(f"Features: {list(X.columns)}")
    
    # Split into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    print(f"\nData split:")
    print(f"Training set: {X_train.shape[0]} patients")
    print(f"Test set: {X_test.shape[0]} patients")
    print(f"Training death rate: {y_train.mean():.3f}")
    print(f"Test death rate: {y_test.mean():.3f}")

## 3. Train Multiple Models

In [ ]:
# Define models to compare
print("TRAINING MACHINE LEARNING MODELS")
print("="*40)

models = {
    'Decision Tree': DecisionTreeClassifier(
        random_state=42,
        max_depth=5,
        min_samples_split=10,
        min_samples_leaf=5
    ),
    'Random Forest': RandomForestClassifier(
        random_state=42,
        n_estimators=100,
        max_depth=5,
        min_samples_split=10
    ),
    'Logistic Regression': LogisticRegression(
        random_state=42,
        max_iter=1000
    )
}

# Train each model and store results
trained_models = {}
cv_scores = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Train model
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    # Cross-validation
    cv_score = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    cv_scores[name] = cv_score
    
    print(f"Cross-validation accuracy: {cv_score.mean():.3f} (+/- {cv_score.std() * 2:.3f})")

print("\nAll models trained successfully!")

## 4. Evaluate Models on Test Set

In [ ]:
# Test all models and compare to hospital baseline
print("EVALUATING MODELS ON TEST SET")
print("="*40)

HOSPITAL_BASELINE = 0.839
results = {}

for name, model in trained_models.items():
    # Make predictions
    y_pred = model.predict(X_test)
    
    # Calculate accuracy
    accuracy = accuracy_score(y_test, y_pred)
    improvement = accuracy - HOSPITAL_BASELINE
    
    results[name] = {
        'accuracy': accuracy,
        'improvement': improvement,
        'predictions': y_pred
    }
    
    print(f"\n{name}:")
    print(f"  Accuracy: {accuracy:.3f} ({accuracy:.1%})")
    print(f"  Hospital baseline: {HOSPITAL_BASELINE:.3f} ({HOSPITAL_BASELINE:.1%})")
    print(f"  Improvement: {improvement:+.3f} ({improvement:+.1%})")
    print(f"  Beats baseline: {'YES' if accuracy > HOSPITAL_BASELINE else 'NO'}")

## 5. Select and Save Best Model

In [ ]:
# Find best performing model
print("SELECTING BEST MODEL")
print("="*40)

best_accuracy = 0
best_model_name = None

for name, result in results.items():
    if result['accuracy'] > best_accuracy:
        best_accuracy = result['accuracy']
        best_model_name = name

best_model = trained_models[best_model_name]

print(f"Best model: {best_model_name}")
print(f"Best accuracy: {best_accuracy:.3f} ({best_accuracy:.1%})")
print(f"Improvement over hospital: {best_accuracy - HOSPITAL_BASELINE:+.3f}")

# Save the best model
try:
    with open('../best_model.pkl', 'wb') as f:
        pickle.dump(best_model, f)
    print(f"\nModel saved as 'best_model.pkl'")
except Exception as e:
    print(f"Error saving model: {e}")

## 6. Model Performance Visualization

In [ ]:
# Create performance comparison chart
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Model accuracy comparison
model_names = list(results.keys())
accuracies = [results[name]['accuracy'] for name in model_names]

bars = axes[0].bar(model_names, accuracies, color=['skyblue', 'lightgreen', 'lightcoral'])
axes[0].axhline(y=HOSPITAL_BASELINE, color='red', linestyle='--', label='Hospital Baseline (83.9%)')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Model Accuracy Comparison')
axes[0].legend()
axes[0].set_ylim(0.7, 1.0)

# Add accuracy values on bars
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{acc:.3f}', ha='center', va='bottom')

# Confusion matrix for best model
best_predictions = results[best_model_name]['predictions']
cm = confusion_matrix(y_test, best_predictions)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
           xticklabels=['Survived', 'Died'],
           yticklabels=['Survived', 'Died'])
axes[1].set_title(f'Confusion Matrix - {best_model_name}')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.suptitle('Week 2 Model Training Results', fontsize=16)
plt.tight_layout()
plt.show()

## 7. Feature Importance Analysis

In [ ]:
# Analyze feature importance (for tree-based models)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*40)

if hasattr(best_model, 'feature_importances_'):
    # Get feature importance
    importances = best_model.feature_importances_
    
    # Create feature importance dataframe
    feature_df = pd.DataFrame({
        'Feature': X_train.columns,
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    
    print("Top 5 most important features:")
    for i, (_, row) in enumerate(feature_df.head(5).iterrows(), 1):
        print(f"  {i}. {row['Feature']}: {row['Importance']:.3f}")
    
    # Plot feature importance
    plt.figure(figsize=(10, 6))
    top_features = feature_df.head(8)
    plt.barh(range(len(top_features)), top_features['Importance'][::-1])
    plt.yticks(range(len(top_features)), top_features['Feature'][::-1])
    plt.xlabel('Feature Importance')
    plt.title(f'Feature Importance - {best_model_name}')
    plt.tight_layout()
    plt.show()
    
else:
    print(f"Feature importance not available for {best_model_name}")

## 8. Week 2 Summary and Handoff

In [ ]:
# Final summary
print("WEEK 2 SUMMARY")
print("="*50)

print(f"Models trained: {len(models)}")
print(f"Best model: {best_model_name}")
print(f"Best accuracy: {best_accuracy:.3f} ({best_accuracy:.1%})")
print(f"Hospital baseline: {HOSPITAL_BASELINE:.3f} ({HOSPITAL_BASELINE:.1%})")

if best_accuracy > HOSPITAL_BASELINE:
    print(f"SUCCESS: Beat baseline by {best_accuracy - HOSPITAL_BASELINE:+.3f} ({(best_accuracy - HOSPITAL_BASELINE) * 100:+.1f}%)")
else:
    print(f"MISS: Below baseline by {best_accuracy - HOSPITAL_BASELINE:.3f} ({(best_accuracy - HOSPITAL_BASELINE) * 100:.1f}%)")

print(f"\nModel saved for Week 3 evaluation")
print("Next: Nasiru and Jose will thoroughly test the model")

# Save results summary
summary = {
    'best_model': best_model_name,
    'accuracy': best_accuracy,
    'baseline': HOSPITAL_BASELINE,
    'improvement': best_accuracy - HOSPITAL_BASELINE,
    'success': best_accuracy > HOSPITAL_BASELINE
}

try:
    with open('../week2_results.txt', 'w') as f:
        for key, value in summary.items():
            f.write(f"{key}: {value}\n")
    print("Results summary saved to 'week2_results.txt'")
except Exception as e:
    print(f"Error saving summary: {e}")

print("\nREADY FOR WEEK 3: Model Evaluation and Presentation")